In [ ]:
# Cell 1 - Imports
# =================
# Import Required Libraries
import os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Configuration
IMG_HEIGHT = 28
IMG_WIDTH = 28

# Paths
BASE_DIR = os.path.dirname(os.getcwd())
MODEL_PATH = os.path.join(BASE_DIR, 'model.keras')
TEST_DIR = os.path.join(BASE_DIR, 'test')
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
RESULT_PATH = os.path.join(BASE_DIR, 'result.csv')

print(f"Model path: {MODEL_PATH}")
print(f"Test directory: {TEST_DIR}")
print(f"Results will be saved to: {RESULT_PATH}")

## 1. Load Trained Model

In [ ]:
# Load the trained model
if os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully!")
    print(f"Input shape: {model.input_shape}")
    print(f"Output shape: {model.output_shape}")
else:
    print(f"Model not found at {MODEL_PATH}")
    print("Please run model_training.ipynb first to train the model.")

In [ ]:
# Get class names from training directory
class_names = None
if os.path.exists(TRAIN_DIR):
    class_names = sorted([d for d in os.listdir(TRAIN_DIR) 
                         if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    print(f"Classes: {class_names}")
else:
    print("Training directory not found. Using numeric labels.")

## 2. Test on Individual Images

In [ ]:
def preprocess_image(img_path):
    """
    Preprocess a single image for prediction
    """
    img = image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode='grayscale')
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array, img

def predict_image(model, img_path, class_names=None):
    """
    Make prediction on a single image and display results
    """
    img_array, original_img = preprocess_image(img_path)
    
    # Predict
    predictions = model.predict(img_array, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = np.max(predictions[0])
    
    # Get label
    if class_names:
        label = class_names[predicted_class]
    else:
        label = str(predicted_class)
    
    # Display results
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Original image
    axes[0].imshow(original_img, cmap='gray')
    axes[0].set_title(f'Predicted: {label}\nConfidence: {confidence:.4f}')
    axes[0].axis('off')
    
    # Prediction probabilities
    if class_names:
        axes[1].barh(class_names, predictions[0], color='steelblue')
    else:
        axes[1].barh(range(len(predictions[0])), predictions[0], color='steelblue')
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Class Probabilities')
    axes[1].set_xlim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    return label, confidence, predictions[0]

print("Prediction functions defined")

In [ ]:
# Test on a sample image
test_images = glob.glob(os.path.join(TEST_DIR, '**', '*.*'), recursive=True)
test_images = [f for f in test_images if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

if test_images and 'model' in dir():
    print(f"Found {len(test_images)} test images")
    # Test on first image
    label, conf, probs = predict_image(model, test_images[0], class_names)
    print(f"\nFile: {os.path.basename(test_images[0])}")
    print(f"Prediction: {label} (confidence: {conf:.4f})")
else:
    print("No test images found or model not loaded")

## 3. Batch Prediction on Test Set

In [ ]:
def batch_predict(model, test_dir, class_names=None):
    """
    Make predictions on all images in the test directory
    """
    results = []
    
    # Get all image files
    image_files = glob.glob(os.path.join(test_dir, '**', '*.*'), recursive=True)
    image_files = [f for f in image_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
    
    if not image_files:
        print(f"No images found in {test_dir}")
        return None
    
    print(f"Processing {len(image_files)} images...")
    
    for img_path in image_files:
        try:
            img_array, _ = preprocess_image(img_path)
            predictions = model.predict(img_array, verbose=0)
            predicted_class = np.argmax(predictions[0])
            confidence = np.max(predictions[0])
            
            if class_names:
                label = class_names[predicted_class]
            else:
                label = str(predicted_class)
            
            rel_path = os.path.relpath(img_path, test_dir)
            
            results.append({
                'image_path': rel_path,
                'predicted_label': label,
                'confidence': confidence,
                'predicted_class_id': predicted_class
            })
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            results.append({
                'image_path': os.path.relpath(img_path, test_dir),
                'predicted_label': 'ERROR',
                'confidence': 0.0,
                'predicted_class_id': -1
            })
    
    return pd.DataFrame(results)

print("Batch prediction function defined")

In [ ]:
# Run batch prediction
if 'model' in dir() and os.path.exists(TEST_DIR):
    results_df = batch_predict(model, TEST_DIR, class_names)
    
    if results_df is not None and len(results_df) > 0:
        print("\nPrediction Results:")
        print("=" * 60)
        print(results_df.to_string(index=False))
        
        # Summary statistics
        print("\n" + "=" * 60)
        print("Summary Statistics:")
        print(f"Total images: {len(results_df)}")
        print(f"Average confidence: {results_df['confidence'].mean():.4f}")
        print(f"Min confidence: {results_df['confidence'].min():.4f}")
        print(f"Max confidence: {results_df['confidence'].max():.4f}")
else:
    print("Model not loaded or test directory not found")

## 4. Export Results to CSV

In [ ]:
# Save results to CSV
if 'results_df' in dir() and results_df is not None:
    results_df.to_csv(RESULT_PATH, index=False)
    print(f"Results saved to: {RESULT_PATH}")
    
    # Display saved file
    print("\nSaved CSV contents:")
    print(pd.read_csv(RESULT_PATH).to_string())
else:
    print("No results to save")

## 5. Interactive Testing

In [ ]:
def display_predictions_grid(model, image_paths, class_names=None, cols=4):
    """
    Display predictions for multiple images in a grid
    """
    n_images = len(image_paths)
    rows = (n_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten() if n_images > 1 else [axes]
    
    for i, img_path in enumerate(image_paths):
        try:
            img_array, original_img = preprocess_image(img_path)
            predictions = model.predict(img_array, verbose=0)
            predicted_class = np.argmax(predictions[0])
            confidence = np.max(predictions[0])
            
            if class_names:
                label = class_names[predicted_class]
            else:
                label = str(predicted_class)
            
            axes[i].imshow(original_img, cmap='gray')
            axes[i].set_title(f'{label}\n({confidence:.2%})', fontsize=10)
            axes[i].axis('off')
            
        except Exception as e:
            axes[i].text(0.5, 0.5, 'Error', ha='center', va='center')
            axes[i].axis('off')
    
    # Hide empty subplots
    for i in range(n_images, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('predictions_grid.png', dpi=150)
    plt.show()

# Display predictions for test images
if test_images and 'model' in dir():
    display_predictions_grid(model, test_images[:16], class_names)
else:
    print("No test images found or model not loaded")

In [ ]:
# Test on a specific image path (modify this path)
# custom_image_path = "path/to/your/image.png"
# if os.path.exists(custom_image_path) and 'model' in dir():
#     predict_image(model, custom_image_path, class_names)

## Summary

In this notebook, we have:
1. Loaded the trained model
2. Tested predictions on individual images
3. Performed batch predictions on the test set
4. Exported results to CSV
5. Created visualization tools for predictions

The model is now ready for deployment. You can use the `run.py` script for command-line deployment.